# BRIO with IndoBART-v2 on Liputan6

This notebook sets up the necessary environments, patches legacy packages for IndoBart, generates candidates, preprocesses the Liputan6 data, and begins tuning.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!git clone https://github.com/alvrian/BRIO.git
%cd BRIO
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
!conda create --name env --file spec-file.txt -y
!conda run -n env pip install -r requirements.txt
!conda run -n env pip install --upgrade transformers torch requests

In [ ]:
%%writefile patch_indobart_legacy.py
import os
print('Patching legacy MBART mapping for IndoBART-v2')
with open('modeling_bart.py', 'r', encoding='utf-8') as f:
    content = f.read()
content = content.replace('BartEncoder', 'MBartEncoder')
content = content.replace('BartDecoder', 'MBartDecoder')
content = content.replace('BartModel', 'MBartModel')
with open('modeling_mbart.py', 'w', encoding='utf-8') as f:
    f.write(content)
with open('model.py', 'r', encoding='utf-8') as f:
    content = f.read()
content = content.replace('from transformers import BartTokenizer, BartModel', 'from transformers import MBartTokenizerFast, MBartForConditionalGeneration, BartModel')
with open('model.py', 'w', encoding='utf-8') as f:
    f.write(content)
print('Patch applied successfully.')

In [ ]:
!conda run -n env python patch_indobart_legacy.py

In [ ]:
!conda run -n env python convert_liputan6_brio.py --input liputan6_sample.json --output_dir liputan6_processed --split test
!conda run -n env python convert_liputan6_brio.py --input liputan6_sample.json --output_dir liputan6_processed --split train
!conda run -n env python convert_liputan6_brio.py --input liputan6_sample.json --output_dir liputan6_processed --split val

In [ ]:
!conda run -n env python gen_candidate.py --gpuid 0 --src_dir ./liputan6_processed/test.source --tgt_dir ./liputan6_processed/test.out --dataset liputan6
!conda run -n env python gen_candidate.py --gpuid 0 --src_dir ./liputan6_processed/train.source --tgt_dir ./liputan6_processed/train.out --dataset liputan6
!conda run -n env python gen_candidate.py --gpuid 0 --src_dir ./liputan6_processed/val.source --tgt_dir ./liputan6_processed/val.out --dataset liputan6

In [ ]:
# Bypass Stanford CoreNLP tokenization for Indonesian by duplicating files
!cp ./liputan6_processed/test.source ./liputan6_processed/test.source.tokenized
!cp ./liputan6_processed/test.target ./liputan6_processed/test.target.tokenized
!cp ./liputan6_processed/test.out ./liputan6_processed/test.out.tokenized
!cp ./liputan6_processed/train.source ./liputan6_processed/train.source.tokenized
!cp ./liputan6_processed/train.target ./liputan6_processed/train.target.tokenized
!cp ./liputan6_processed/train.out ./liputan6_processed/train.out.tokenized
!cp ./liputan6_processed/val.source ./liputan6_processed/val.source.tokenized
!cp ./liputan6_processed/val.target ./liputan6_processed/val.target.tokenized
!cp ./liputan6_processed/val.out ./liputan6_processed/val.out.tokenized

In [ ]:
!conda run -n env python preprocess.py --src_dir ./liputan6_processed --tgt_dir ./liputan6_brio --split test --cand_num 16 --dataset liputan6 -l
!conda run -n env python preprocess.py --src_dir ./liputan6_processed --tgt_dir ./liputan6_brio --split train --cand_num 16 --dataset liputan6 -l
!conda run -n env python preprocess.py --src_dir ./liputan6_processed --tgt_dir ./liputan6_brio --split val --cand_num 16 --dataset liputan6 -l

In [ ]:
!conda run -n env python main.py --cuda --gpuid 0 --config cnndm -l --model_type indobenchmark/indobart-v2 --dataset liputan6